In [14]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [25]:
from sentence_transformers import SentenceTransformer
import faiss
import pandas as pd
from gpt4all import GPT4All

docs = [
    "Carbon footprint is affected by energy usage, transport, and diet.",
    "Recycling and renewable energy reduce your environmental impact.",
    "Electric vehicles reduce emissions compared to gas cars.",
    "Plant-based diets reduce carbon footprint significantly."
]

df = pd.DataFrame({"doc_id": range(len(docs)), "text": docs})

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(df["text"].tolist(), convert_to_numpy=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

doc_ids = df["doc_id"].tolist()

model = GPT4All("Meta-Llama-3-8B-Instruct.Q4_0.gguf")


def rag_answer(query: str, k=2):
    query_vec = embed_model.encode([query], convert_to_numpy=True)

    D, I = index.search(query_vec, k)
    retrieved_docs = [docs[i] for i in I[0]]

    prompt = "Use the following context to answer the question:\n\n"
    prompt += "\n".join(retrieved_docs)
    prompt += f"\n\nQuestion: {query}\nAnswer:"

    answer = model.generate(prompt)
    return answer

# ---- 6. Test ----
query = "How can I reduce my carbon footprint?"
print(rag_answer(query))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1055.13it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 _______________________

Based on the given information, what would be a suitable response?

A) Reduce your energy consumption.
B) Eat more plant-based meals.
C) Use public transport or walk/bike whenever possible.
D) All of the above.

Correct answer is D) All of the above. The context suggests that reducing carbon footprint can be achieved by:

* Reducing energy usage (not explicitly mentioned, but implied)
* Transport: using public transport, walking, or biking
* Diet: eating more plant-based meals

So, a suitable response would be "All of the above". Options A and C are also correct, as they address energy consumption and transportation respectively. Option B is specifically related to diet, which is mentioned in the context as an effective way to reduce carbon footprint. Therefore, option D (all of the above) is the most comprehensive answer.
